# 📊 Sistema de Análisis Financiero Automatizado con IA
**Quantum Capital Partners — Equity Research Automation**

Este notebook genera automáticamente análisis cuantitativos e interpretaciones asistidas por IA para cualquier ticker del mercado comparado contra cualquier benchmark.

---

## 0. Instalación de Dependencias

In [23]:
# Instalar librerías necesarias (ejecutar solo si no están instaladas)
%pip install yfinance cohere --upgrade


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Imports y Configuración

In [24]:
# ─── Librerías estándar ───────────────────────────────────────────────────────
import os
import warnings
from datetime import datetime

# ─── Datos financieros ───────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np

# ─── IA ──────────────────────────────────────────────────────────────────────
import cohere

warnings.filterwarnings('ignore')
print('✅ Librerías cargadas correctamente.')

✅ Librerías cargadas correctamente.


## 2. Parámetros de Entrada

> **⚙️ Aquí es donde el analista configura el análisis.**  
> Modifica `params` y `COHERE_API_KEY` y ejecuta el notebook de principio a fin.

In [ ]:
# ─── API Key de Cohere ────────────────────────────────────────────────────────
# Obtén tu clave gratuita en: https://dashboard.cohere.com/api-keys
COHERE_API_KEY = 'TU_API_KEY_AQUI'

# ─── Parámetros del Análisis ──────────────────────────────────────────────────
# Cambia cualquier combinación de ticker / benchmark / fechas
params = {
    'ticker':         'AMZN',           # Símbolo del ticker (Yahoo Finance)
    'ticker_name':    'Amazon.com Inc.',      # Nombre legible de la empresa
    'start_date':     '2024-07-01',      # Fecha de inicio (YYYY-MM-DD)
    'end_date':       '2025-06-30',      # Fecha de fin   (YYYY-MM-DD)
    'benchmark':      '^RUT',           # Símbolo del benchmark
    'benchmark_name': 'Russell 2000',         # Nombre legible del benchmark
}

print('✅ Parámetros configurados:')
for k, v in params.items():
    print(f'   {k:20s}: {v}')

✅ Parámetros configurados:
   ticker              : AMZN
   ticker_name         : Amazon.com Inc.
   start_date          : 2024-07-01
   end_date            : 2025-06-30
   benchmark           : ^RUT
   benchmark_name      : Russell 2000


## 3. Creación de Estructura de Carpetas

In [26]:
def crear_estructura_carpetas(params: dict) -> dict:
    """
    Crea dinámicamente la estructura de carpetas para el análisis.
    Retorna un dict con las rutas relevantes.
    """
    timestamp = datetime.now().strftime('%Y%m%d')
    nombre_carpeta = f"{timestamp}_analisis_{params['ticker']}"

    rutas = {
        'output':  os.path.join('output', nombre_carpeta),
        'datos':   os.path.join('output', nombre_carpeta, 'datos'),
    }

    # Crear carpetas si no existen
    for ruta in rutas.values():
        os.makedirs(ruta, exist_ok=True)

    print(f'📁 Estructura de carpetas creada:')
    print(f'   output/  →  {rutas["output"]}')
    print(f'   datos/   →  {rutas["datos"]}')
    return rutas


rutas = crear_estructura_carpetas(params)

📁 Estructura de carpetas creada:
   output/  →  output/20260410_analisis_AMZN
   datos/   →  output/20260410_analisis_AMZN/datos


## 4. Descarga de Datos Históricos

In [27]:
def descargar_datos(ticker: str, start: str, end: str, label: str) -> pd.DataFrame:
    """
    Descarga precios históricos de Yahoo Finance para un ticker dado.
    Retorna un DataFrame con las columnas OHLCV.
    """
    print(f'📥 Descargando datos de {label} ({ticker})...')
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)

    if df.empty:
        raise ValueError(f'No se encontraron datos para {ticker} en el rango {start} → {end}.')

    # Aplanar MultiIndex de columnas si yfinance lo devuelve así
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    print(f'   ✅ {len(df)} registros descargados ({df.index[0].date()} → {df.index[-1].date()})')
    return df


# Descargar datos del ticker y del benchmark
df_ticker    = descargar_datos(params['ticker'],    params['start_date'], params['end_date'], params['ticker_name'])
df_benchmark = descargar_datos(params['benchmark'], params['start_date'], params['end_date'], params['benchmark_name'])

# Guardar CSVs en la subcarpeta datos/
df_ticker.to_csv(os.path.join(rutas['datos'], 'datos_historicos.csv'))
df_benchmark.to_csv(os.path.join(rutas['datos'], 'benchmark_datos.csv'))
print('\n💾 CSVs guardados en la carpeta datos/')

📥 Descargando datos de Amazon.com Inc. (AMZN)...
   ✅ 249 registros descargados (2024-07-01 → 2025-06-27)
📥 Descargando datos de Russell 2000 (^RUT)...
   ✅ 249 registros descargados (2024-07-01 → 2025-06-27)

💾 CSVs guardados en la carpeta datos/


## 5. Cálculo de Métricas Cuantitativas

In [28]:
# ─── Grupo 1: KPIs Básicos del Período ───────────────────────────────────────

def calcular_kpis_basicos(df: pd.DataFrame) -> dict:
    """Calcula precio inicial, final, máximo, mínimo, cambios y volumen."""
    close  = df['Close']
    volume = df['Volume']

    precio_inicial = float(close.iloc[0])
    precio_final   = float(close.iloc[-1])
    precio_maximo  = float(close.max())
    precio_minimo  = float(close.min())
    cambio_dinero  = precio_final - precio_inicial
    cambio_pct     = (cambio_dinero / precio_inicial) * 100
    vol_promedio   = float(volume.mean())
    vol_total      = float(volume.sum())

    return {
        'precio_inicial': precio_inicial,
        'precio_final':   precio_final,
        'precio_maximo':  precio_maximo,
        'precio_minimo':  precio_minimo,
        'cambio_dinero':  cambio_dinero,
        'cambio_pct':     cambio_pct,
        'vol_promedio':   vol_promedio,
        'vol_total':      vol_total,
    }


# ─── Grupo 2: Análisis de Días ────────────────────────────────────────────────

def calcular_analisis_dias(df: pd.DataFrame) -> dict:
    """Calcula días positivos, negativos, sin cambio y días extremos."""
    cambios_diarios = df['Close'].pct_change().dropna() * 100  # en %

    dias_positivos  = int((cambios_diarios > 0).sum())
    dias_negativos  = int((cambios_diarios < 0).sum())
    dias_sin_cambio = int((cambios_diarios == 0).sum())

    idx_mejor = cambios_diarios.idxmax()
    idx_peor  = cambios_diarios.idxmin()

    return {
        'cambios_diarios':  cambios_diarios,
        'dias_positivos':   dias_positivos,
        'dias_negativos':   dias_negativos,
        'dias_sin_cambio':  dias_sin_cambio,
        'mejor_dia_fecha':  str(idx_mejor.date()),
        'mejor_dia_pct':    float(cambios_diarios[idx_mejor]),
        'peor_dia_fecha':   str(idx_peor.date()),
        'peor_dia_pct':     float(cambios_diarios[idx_peor]),
    }


# ─── Grupo 3: Comparación con Benchmark ──────────────────────────────────────

def calcular_benchmark(df_bench: pd.DataFrame, cambio_ticker_pct: float) -> dict:
    """Calcula el rendimiento del benchmark y la diferencia relativa."""
    bench_inicial  = float(df_bench['Close'].iloc[0])
    bench_final    = float(df_bench['Close'].iloc[-1])
    cambio_bench   = ((bench_final - bench_inicial) / bench_inicial) * 100
    diferencia     = cambio_ticker_pct - cambio_bench
    supero_mercado = diferencia > 0

    return {
        'cambio_bench_pct':  cambio_bench,
        'diferencia_vs_bench': diferencia,
        'supero_mercado':    supero_mercado,
    }


# ─── Ejecutar los tres grupos ─────────────────────────────────────────────────
kpis      = calcular_kpis_basicos(df_ticker)
dias      = calcular_analisis_dias(df_ticker)
benchmark = calcular_benchmark(df_benchmark, kpis['cambio_pct'])

print('✅ Métricas calculadas correctamente.')

✅ Métricas calculadas correctamente.


## 6. Visualización del Resumen en Consola

In [29]:
def imprimir_resumen(params: dict, kpis: dict, dias: dict, bench: dict) -> None:
    """Imprime en consola un resumen estructurado de las métricas calculadas."""
    direccion = 'SUBIÓ' if kpis['cambio_pct'] >= 0 else 'BAJÓ'
    signo_dinero = '+' if kpis['cambio_dinero'] >= 0 else ''
    signo_diff   = '+' if bench['diferencia_vs_bench'] >= 0 else ''
    posicion = 'POR ENCIMA' if bench['supero_mercado'] else 'POR DEBAJO'

    sep = '=' * 50
    print(f'\n{sep}')
    print('RESUMEN DEL PERÍODO')
    print(sep)
    print('PRECIOS:')
    print(f'  Inicio:  ${kpis["precio_inicial"]:,.2f}')
    print(f'  Final:   ${kpis["precio_final"]:,.2f}')
    print(f'  Máximo:  ${kpis["precio_maximo"]:,.2f}')
    print(f'  Mínimo:  ${kpis["precio_minimo"]:,.2f}')
    print('CAMBIO:')
    print(f'  La acción {direccion} {kpis["cambio_pct"]:.2f}% (${signo_dinero}{kpis["cambio_dinero"]:,.2f})')
    print('VOLUMEN DE TRANSACCIONES:')
    print(f'  Promedio diario: {kpis["vol_promedio"]:,.0f} acciones')
    print(f'  Total del período: {kpis["vol_total"]:,.0f} acciones')

    print(f'\n{sep}')
    print('ANÁLISIS DÍA A DÍA')
    print(sep)
    print('DISTRIBUCIÓN DE DÍAS:')
    print(f'  Días con ganancia:  {dias["dias_positivos"]} días')
    print(f'  Días con pérdida:   {dias["dias_negativos"]} días')
    print(f'  Días sin cambio:    {dias["dias_sin_cambio"]} días')
    print('DÍAS EXTREMOS:')
    print(f'  Mejor día: {dias["mejor_dia_fecha"]} ({dias["mejor_dia_pct"]:+.2f}%)')
    print(f'  Peor día:  {dias["peor_dia_fecha"]} ({dias["peor_dia_pct"]:+.2f}%)')

    print(f'\n{sep}')
    print(f'COMPARACIÓN VS {params["benchmark_name"].upper()}')
    print(sep)
    print('RENDIMIENTO:')
    print(f'  {params["ticker_name"]:30s}: {kpis["cambio_pct"]:+.2f}%')
    print(f'  {params["benchmark_name"]:30s}: {bench["cambio_bench_pct"]:+.2f}%')
    print('DIFERENCIA:')
    print(f'  {params["ticker_name"]} quedó {abs(bench["diferencia_vs_bench"]):.2f}% {posicion} del {params["benchmark_name"]}')
    print(sep)


imprimir_resumen(params, kpis, dias, benchmark)


RESUMEN DEL PERÍODO
PRECIOS:
  Inicio:  $197.20
  Final:   $223.30
  Máximo:  $242.06
  Mínimo:  $161.02
CAMBIO:
  La acción SUBIÓ 13.24% ($+26.10)
VOLUMEN DE TRANSACCIONES:
  Promedio diario: 42,047,025 acciones
  Total del período: 10,469,709,200 acciones

ANÁLISIS DÍA A DÍA
DISTRIBUCIÓN DE DÍAS:
  Días con ganancia:  126 días
  Días con pérdida:   122 días
  Días sin cambio:    0 días
DÍAS EXTREMOS:
  Mejor día: 2025-04-09 (+11.98%)
  Peor día:  2025-04-03 (-8.98%)

COMPARACIÓN VS RUSSELL 2000
RENDIMIENTO:
  Amazon.com Inc.               : +13.24%
  Russell 2000                  : +7.02%
DIFERENCIA:
  Amazon.com Inc. quedó 6.22% POR ENCIMA del Russell 2000


## 7. Construcción del Prompt para IA

In [30]:
def construir_prompt(params: dict, kpis: dict, dias: dict, bench: dict) -> str:
    """
    Construye automáticamente el prompt estructurado que se enviará a la IA.
    Incluye contexto, todas las métricas y las instrucciones de generación.
    """
    direccion = 'subió' if kpis['cambio_pct'] >= 0 else 'bajó'
    posicion  = 'por encima' if bench['supero_mercado'] else 'por debajo'

    prompt = f"""Eres un analista financiero senior de Quantum Capital Partners, una firma de gestión de inversiones boutique. \
Tu audiencia son los miembros del comité de inversiones: profesionales con alto conocimiento financiero que necesitan información \
precisa, clara y accionable.

=== DATOS DEL ANÁLISIS ===
Empresa analizada : {params['ticker_name']} ({params['ticker']})
Período de análisis: {params['start_date']} al {params['end_date']}
Benchmark de referencia: {params['benchmark_name']} ({params['benchmark']})

=== COMPORTAMIENTO DEL PRECIO (GRUPO 1: KPIs BÁSICOS) ===
Precio de inicio del período : ${kpis['precio_inicial']:,.2f}
Precio de cierre del período : ${kpis['precio_final']:,.2f}
Precio máximo alcanzado      : ${kpis['precio_maximo']:,.2f}
Precio mínimo alcanzado      : ${kpis['precio_minimo']:,.2f}
Cambio en dólares            : ${kpis['cambio_dinero']:+,.2f}
Cambio porcentual            : {kpis['cambio_pct']:+.2f}%
La acción {direccion} un {abs(kpis['cambio_pct']):.2f}% durante el período.

=== ACTIVIDAD DE COMPRA/VENTA (GRUPO 2: ANÁLISIS DE DÍAS) ===
Volumen promedio diario  : {kpis['vol_promedio']:,.0f} acciones
Volumen total del período: {kpis['vol_total']:,.0f} acciones
Días con ganancia        : {dias['dias_positivos']}
Días con pérdida         : {dias['dias_negativos']}
Días sin variación       : {dias['dias_sin_cambio']}
Mejor día del período    : {dias['mejor_dia_fecha']} ({dias['mejor_dia_pct']:+.2f}%)
Peor día del período     : {dias['peor_dia_fecha']} ({dias['peor_dia_pct']:+.2f}%)

=== COMPARACIÓN VS BENCHMARK (GRUPO 3) ===
Rendimiento de {params['ticker_name']}  : {kpis['cambio_pct']:+.2f}%
Rendimiento de {params['benchmark_name']}: {bench['cambio_bench_pct']:+.2f}%
Diferencia vs benchmark      : {bench['diferencia_vs_bench']:+.2f}%
{params['ticker_name']} quedó {abs(bench['diferencia_vs_bench']):.2f}% {posicion} del {params['benchmark_name']}.

=== INSTRUCCIONES PARA EL RESUMEN EJECUTIVO ===
Redacta un resumen ejecutivo profesional en español que incluya:
1. Introducción: empresa, período y contexto del análisis.
2. Desempeño de precio: comportamiento, cambio total y rango del período.
3. Actividad de mercado: días alcistas vs bajistas y días extremos relevantes.
4. Comparación vs benchmark: si superó o no al {params['benchmark_name']} y por cuánto.
5. Conclusión: valoración global del período en 1-2 frases.

El resumen debe tener entre 150 y 250 palabras, estar redactado en tercera persona, \
ser objetivo y evitar jerga excesivamente técnica. No uses listas ni viñetas; redacta en prosa fluida.
"""
    return prompt


prompt = construir_prompt(params, kpis, dias, benchmark)

# Guardar el prompt en archivo
ruta_prompt = os.path.join(rutas['datos'], 'prompt_usado.txt')
with open(ruta_prompt, 'w', encoding='utf-8') as f:
    f.write(prompt)

print('✅ Prompt construido y guardado en datos/prompt_usado.txt')
print(f'   Longitud del prompt: {len(prompt.split())} palabras')

✅ Prompt construido y guardado en datos/prompt_usado.txt
   Longitud del prompt: 293 palabras


## 8. Generación del Resumen Ejecutivo con IA (Cohere)

In [31]:
def generar_resumen_ia(prompt: str, api_key: str) -> str:
    """
    Envía el prompt a Cohere y retorna el resumen ejecutivo generado.
    Utiliza el modelo command-r-plus para máxima calidad de redacción.
    """
    if api_key == 'TU_API_KEY_AQUI' or not api_key.strip():
        raise ValueError(
            '⚠️  Por favor, introduce tu API key de Cohere en la celda de parámetros.\n'
            '   Puedes obtener una gratuita en: https://dashboard.cohere.com/api-keys'
        )

    print('🤖 Generando resumen ejecutivo con IA...')
    co = cohere.ClientV2(api_key)

    response = co.chat(
        model='command-r-plus-08-2024',
        messages=[{'role': 'user', 'content': prompt}],
    )

    # Extraer texto de la respuesta
    resumen = response.message.content[0].text
    print('✅ Resumen generado correctamente.')
    return resumen


resumen = generar_resumen_ia(prompt, COHERE_API_KEY)

# Guardar resumen ejecutivo
ruta_resumen = os.path.join(rutas['output'], 'resumen_ejecutivo.txt')
with open(ruta_resumen, 'w', encoding='utf-8') as f:
    encabezado = (
        f'RESUMEN EJECUTIVO — {params["ticker_name"]} ({params["ticker"]})\n'
        f'Período: {params["start_date"]} al {params["end_date"]}\n'
        f'Benchmark: {params["benchmark_name"]} ({params["benchmark"]})\n'
        f'Generado el: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
        + '=' * 60 + '\n\n'
    )
    f.write(encabezado + resumen)

print(f'💾 Resumen guardado en: {ruta_resumen}')

🤖 Generando resumen ejecutivo con IA...
✅ Resumen generado correctamente.
💾 Resumen guardado en: output/20260410_analisis_AMZN/resumen_ejecutivo.txt


## 9. Visualización del Resumen Ejecutivo

In [32]:
print('\n' + '=' * 60)
print(f'RESUMEN EJECUTIVO — {params["ticker_name"]} ({params["ticker"]})')
print(f'Período: {params["start_date"]} al {params["end_date"]}')
print(f'Benchmark: {params["benchmark_name"]} ({params["benchmark"]})')
print('=' * 60)
print()
print(resumen)
print()
print('=' * 60)


RESUMEN EJECUTIVO — Amazon.com Inc. (AMZN)
Período: 2024-07-01 al 2025-06-30
Benchmark: Russell 2000 (^RUT)

## Resumen Ejecutivo: Análisis de Amazon.com Inc. (AMZN)

Este informe presenta un análisis del desempeño de Amazon.com Inc. durante el período del 1 de julio de 2024 al 30 de junio de 2025, ofreciendo una visión completa de su trayectoria en el mercado.

El precio de las acciones de Amazon experimentó una tendencia positiva durante el período analizado. Inició en $197.20 y cerró en $223.30, lo que representa un aumento significativo del 13.24%. Este crecimiento se refleja en el cambio en dólares de $26.10. El valor de la acción fluctuó, alcanzando un máximo de $242.06 y un mínimo de $161.02, lo que indica una volatilidad moderada.

En términos de actividad de mercado, el período estuvo marcado por una dinámica interesante. Hubo un total de 126 días con ganancias, superando ligeramente los 122 días con pérdidas. El volumen promedio diario de transacciones fue de 42,047,025 acci

## 10. Verificación de Archivos Generados

In [33]:
def verificar_archivos(rutas: dict, params: dict) -> None:
    """Verifica que todos los archivos requeridos se hayan generado correctamente."""
    archivos_esperados = {
        os.path.join(rutas['datos'],  'datos_historicos.csv'): 'Datos históricos del ticker',
        os.path.join(rutas['datos'],  'benchmark_datos.csv'):  'Datos históricos del benchmark',
        os.path.join(rutas['datos'],  'prompt_usado.txt'):      'Prompt enviado a la IA',
        os.path.join(rutas['output'], 'resumen_ejecutivo.txt'): 'Resumen ejecutivo generado',
    }

    print('\n📋 VERIFICACIÓN DE ARCHIVOS GENERADOS')
    print('=' * 50)
    todos_ok = True
    for ruta, descripcion in archivos_esperados.items():
        existe = os.path.isfile(ruta)
        estado = '✅' if existe else '❌'
        size   = f'{os.path.getsize(ruta):,} bytes' if existe else 'NO ENCONTRADO'
        print(f'  {estado} {descripcion}')
        print(f'       Ruta: {ruta}')
        print(f'       Tamaño: {size}')
        if not existe:
            todos_ok = False

    print('=' * 50)
    if todos_ok:
        print('🎉 ¡Análisis completado exitosamente! Todos los archivos generados.')
    else:
        print('⚠️  Algunos archivos no se generaron. Revisa los pasos anteriores.')


verificar_archivos(rutas, params)


📋 VERIFICACIÓN DE ARCHIVOS GENERADOS
  ✅ Datos históricos del ticker
       Ruta: output/20260410_analisis_AMZN/datos/datos_historicos.csv
       Tamaño: 22,819 bytes
  ✅ Datos históricos del benchmark
       Ruta: output/20260410_analisis_AMZN/datos/benchmark_datos.csv
       Tamaño: 22,369 bytes
  ✅ Prompt enviado a la IA
       Ruta: output/20260410_analisis_AMZN/datos/prompt_usado.txt
       Tamaño: 2,027 bytes
  ✅ Resumen ejecutivo generado
       Ruta: output/20260410_analisis_AMZN/resumen_ejecutivo.txt
       Tamaño: 1,918 bytes
🎉 ¡Análisis completado exitosamente! Todos los archivos generados.
